# SSC4: from T1 training to T2 prediction

Explore a frozen pancreas model trained on T1 MRI. SSC4 compares nearby patches at
radius 1 and supplies four self-similarity channels to a 2D nnU-Net; it has no
raw-intensity feature channel. The comparisons use reflected 3×3 patches.

This notebook shows saved results and predictions. It performs no training;
set `RUN_INFERENCE = False` to browse saved outputs only.


## 1. Open the SSC4 demo

From the repository root:

```sh
uv sync --locked --extra cpu --group ssc --group notebooks
uv run --locked --extra cpu --group ssc --group notebooks jupyter lab notebooks/ssc-domain-generalization.ipynb
```

The first run downloads about **314 MB** of missing files: public SSC4 weights
and saved results from this repository's release, plus six original MRI/reference
pairs directly from pinned CrossPan. **No GitHub login is required.**

Verified local files are reused; a complete bundle works offline. The bundle contains six selected illustrations, all **138 scan scores**,
training curves and the model. Change `DEMO_DIR` below to reuse another location.


In [ ]:
%matplotlib inline
import importlib.metadata as metadata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from ml_biomed_project.ssc import explore
from ml_biomed_project.ssc.download import ensure_demo

PROJECT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/ml_biomed_project").is_dir()
)
DEMO_DIR = PROJECT / "data" / "ssc-demo"
CASE_ID = "NU_T2_126"
RUN_INFERENCE = True
DEVICE = "auto"  # CUDA, then MPS, then CPU.

In [ ]:
ensure_demo(DEMO_DIR)
demo = explore.load_demo(DEMO_DIR)
display(demo.cases)

## 2. Frozen model and training

The model used **323 T1 training cases**, **46 development-validation cases**,
100 epochs and seed 1729. The saved curves show stochastic validation patches;
the results below score complete native volumes. Native compound loss can be negative.


In [ ]:
m = demo.manifest["model"]
display(
    pd.Series(
        {
            "Model": m["method"],
            "Parameters": f"{m['parameters']:,}",
            "Training": f"{m['epochs']} epochs · {m['updates']:,} updates · seed {m['seed']}",
            "Input": f"{m['input_features']} self-similarity channels · patch {m['patch']}",
            "Checkpoint SHA256": m["checkpoint_sha256"],
        }
    ).to_frame("SSC4")
)

In [ ]:
explore.plot_training(pd.read_csv(demo.path("training.csv")))
plt.show()

## 3. Whole-volume results

Source development VAL: **0.6856 mean Dice across 46 scans**. Exploratory T2:
**0.4593 across 85 operational groups** (92 scans; scan mean **0.4695**).
Repeated T2 scans are averaged within each filename-derived group first.
The table and plot use all retained scores, not just the six illustrations.


In [ ]:
display(explore.study_summary(demo.scores).round(4))
explore.plot_study(demo.scores)
plt.show()

## 4. Browse reference and SSC4 prediction

Choose a case, plane and slice. Pink is the reference; teal is SSC4. Dashed pink
contours expose missed reference regions. Scores cover the whole volume.
These are fixed, selected illustrations, not a representative sample.

Controls read saved masks and never rerun inference. Display uses canonical array
coordinates and physical aspect; it does not deoblique images. Thick slices can
make orthogonal views coarse.


In [ ]:
explore.case_viewer(demo, initial_case=CASE_ID)

## 5. Optional one-scan inference

Set `RUN_INFERENCE=True`, then rerun the next cell. Native nnU-Net reads,
normalizes, resamples, tiles, mirrors and restores the original grid. The label is
used only afterwards for scoring. A new output and provenance sidecar are saved;
existing outputs are never overwritten. For another scan, pass its NIfTI path to
`predict_volume`; no reference is needed for prediction.


In [ ]:
if RUN_INFERENCE:
    from ml_biomed_project.ssc import load_model, predict_volume

    model = load_model(demo.model_dir, device=DEVICE)
    row = demo.row(CASE_ID)
    output = PROJECT / "results" / "ssc4-notebook" / f"{CASE_ID}.nii.gz"
    prediction = predict_volume(model, demo.path(row["image"]), output)
    replay = explore.load_case(
        demo.path(row["image"]),
        demo.path(row["reference"]),
        {"SSC4": (prediction, 1)},
        name=CASE_ID,
    )
    display(replay["scores"].round(4))
    explore.plot_case(replay)
    plt.show()
else:
    print("Saved predictions only. Set RUN_INFERENCE=True to predict one scan.")

## Provenance and limits

One initialization; T1 informed development and planning. T2 was previously
exposed during research, and operational groups are not verified patient identities.
This is exploratory transfer evidence, not untouched confirmation or a matched
raw-intensity comparison. The model is not a clinical tool.

Data: [CrossPan](https://arxiv.org/html/2604.18797v1).
Descriptor inspiration: [SSC (2013)](https://iacl.ece.jhu.edu/proceedings/miccai2013/papers/8149/81490187.pdf),
a different 3D registration descriptor. Check the bundle's dataset terms before sharing it.


In [ ]:
pd.Series(
    {
        name: metadata.version(name)
        for name in ["torch", "nnunetv2", "nibabel", "numpy", "matplotlib", "ipywidgets"]
    },
    name="Environment",
)